# Importación de librerías

In [2]:
from pathlib import Path
import json

from peft import PeftModel
from transformers import AutoTokenizer
from unsloth import FastVisionModel

e:\proyecto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu130).
W0522 22:31:33.797000 9580 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
e:\proyecto\.venv\Lib\site-packages\unsloth\__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


Establecemos las rutas para el checkpoint y para la salida con el modelo exportado.

In [6]:
PROJECT_ROOT = Path("E:/proyecto")
ADAPTER_DIR = PROJECT_ROOT / "qwen3-vl-plants" / "checkpoint-175"
OUTPUT_DIR = PROJECT_ROOT / "qwen3-vl-plants" / "qwen3-vl-plant-tuned"

Y establecemos el modelo, tokenizer y tamaño máximo de tokens a utilizar.

In [7]:
BASE_MODEL = "unsloth/Qwen3-VL-4B-Instruct"
BASE_TOKENIZER_MODEL = "Qwen/Qwen3-VL-4B-Instruct"

MAX_SEQ_LENGTH = 4096

# Exportación del modelo

Primero, creamos una función auxiliar para escribir en archivos JSON.

In [ ]:
def write_json(path: Path, data: dict) -> None:
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8"
    )

Y con esto, creamos un método `repair_qwen3_vl_metadata`. En las exportaciones normales de este modelo, al final, terminaba teniendo problemas por "leaking" de los prompts, de forma que salía parte de su prompt al frontend. Es algo relativamente común, que se debe a que los metadatos no guardan los tokens correctos que marcan la finalización de un mensaje, así que el modelo deja de poder diferenciarlos.

Podríamos meternos en bastante detalle con esto, pero básicamente nos interesa cambiar los valores de estos tokens:
*   `eos_token` (end of sequence): token que marca el fin de una secuencia. Lo colocamos como "<|im_end|>".
*   `pad_token` (padding): token que se utiliza para rellenar otras secuencias. Por cómo funciona la GPU, si hay una diferencia con el número de tokens, tratará de rellenarlo con esto. Lo colocamos como "<|endoftext|>".
*   `bos_token` (beggining of sequence): token que marca el inicio de una secuencia. Se deja como `None` a favor de `eos_token`.

In [9]:
def repair_qwen3_vl_metadata(output_dir: Path) -> None:
    """
    Restaura metadatos que necesita LM Studio / GGUF.
    """

    base_tokenizer = AutoTokenizer.from_pretrained(
        BASE_TOKENIZER_MODEL,
        trust_remote_code=True,
    )

    im_end_id = base_tokenizer.convert_tokens_to_ids("<|im_end|>")
    endoftext_id = base_tokenizer.convert_tokens_to_ids("<|endoftext|>")

    tokenizer_config_path = output_dir / "tokenizer_config.json"
    tokenizer_config = json.loads(tokenizer_config_path.read_text(encoding="utf-8"))
    tokenizer_config["chat_template"] = base_tokenizer.chat_template
    tokenizer_config["eos_token"] = "<|im_end|>"
    tokenizer_config["pad_token"] = "<|endoftext|>"
    tokenizer_config["bos_token"] = None
    write_json(tokenizer_config_path, tokenizer_config) # Guarda JSON de configuración de tokenizer.

    config_path = output_dir / "config.json"
    config = json.loads(config_path.read_text(encoding="utf-8"))
    config["eos_token_id"] = im_end_id
    config["pad_token_id"] = endoftext_id
    config["bos_token_id"] = None
    write_json(config_path, config)

    generation_config_path = output_dir / "generation_config.json"
    if generation_config_path.exists():
        generation_config = json.loads(generation_config_path.read_text(encoding="utf-8"))
    else:
        generation_config = {}

    generation_config["eos_token_id"] = [im_end_id, endoftext_id]
    generation_config["pad_token_id"] = endoftext_id
    generation_config["bos_token_id"] = endoftext_id
    write_json(generation_config_path, generation_config) # Guarda JSON de configuración de generación.

    print("Reparados metadatos de Qwen3-VL para exportar como GGUF.")

Ejecutamos el guardado para el modelo en formato HF, haciendo un `merge` de los pesos, guardando el modelo completo y aplicando las reparaciones a los metadatos del método anterior.

In [ ]:
model, processor = FastVisionModel.from_pretrained(
    BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
)

model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
model = model.merge_and_unload()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUTPUT_DIR), safe_serialization=True)
processor.save_pretrained(str(OUTPUT_DIR))

repair_qwen3_vl_metadata(OUTPUT_DIR)
print(f"Guarado modelo en {OUTPUT_DIR}")

==((====))==  Unsloth 2026.5.4: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 12.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights: 100%|██████████| 713/713 [00:01<00:00, 428.19it/s]
[accelerate.big_modeling|WARNING]Some parameters are on the meta device because they were offloaded to the cpu.
Writing model shards: 100%|██████████| 1/1 [00:54<00:00, 54.76s/it]
Unsloth: Restored added_tokens_decoder metadata in E:\proyecto\qwen3-vl-plants\qwen3-vl-plant-tuned\tokenizer_config.json.


Reparados metadatos de Qwen3-VL para exportar como GGUF.
Guarado modelo en E:\proyecto\qwen3-vl-plants\qwen3-vl-plant-tuned


# Instrucciones para exportar hacia GGUF e importar en LM Studio:

*   Descargar `llama.cpp` desde su repositorio oficial.
*   Utilizar el script `convert_hf_to_gguf.py` para hacer lo siguiente:
    *   `py convert_hf_to_gguf.py E:\proyecto\qwen3-vl-plants-lora\qwen3-vl-plant-tuned --outfile E:\proyecto\qwen3-vl-plants-lora\qwen3-vl-plant-tuned.gguf --outtype q8_0`
    *   `py convert_hf_to_gguf.py E:\proyecto\qwen3-vl-plants-lora\qwen3-vl-plant-tuned --outfile E:\proyecto\qwen3-vl-plants-lora\mmproj-model.gguf --outtype q8_0 --mmproj`
*   Tras esto, para importarlo en LM Studio, utilizamos `lms import`.
*   Si ambos archivos GGUF están en la misma carpeta, deberían cargar juntos.